UNIFESP X-Ray — Step 3: Hybrid CNN-Transformer (novelty)

Trains the Step 3 model: DenseNet121 + Swin-Tiny fused via a learned attention gate, with a plain-vs-hierarchical loss ablation (one variant per GPU). This is new for UNIFESP -- no academic paper or Kaggle competition solution we found uses a CNN+Transformer hybrid or a hierarchical anatomical loss on this dataset.

Target to beat (Step 2 baseline, same fold-0 validation protocol):

Swin-Tiny: sample_mean_f1 = 0.928 (closest to the real Kaggle leaderboard metric)
DenseNet121: macro_f1 = 0.875 (best per-class-averaged score)

Before running: attach both datasets (images + pipeline) in Notebook Settings → Data, and turn on GPU T4 x2 in Notebook Settings → Accelerator. If you're continuing in the same session as the Step 2 notebook, you can skip straight to section 3 (training) -- setup and caching are idempotent either way.

1. Environment setup (same auto-detection as Step 2)

In [ ]:
import os

pipeline_root = None
images_root = None

print("Tree under /kaggle/input (dirs only, depth<=2, for debugging -- capped at 200 lines):")
_printed = 0
for cur, dirs, files in os.walk("/kaggle/input"):
    depth = cur.count(os.sep) - "/kaggle/input".count(os.sep)
    if depth <= 2 and _printed < 200:
        print("  " * depth + os.path.basename(cur) + "/")
        _printed += 1
    elif _printed == 200:
        print("  ... (truncated, too many entries -- this is fine, detection below doesn't depend on this print)")
        _printed += 1

    if "src" in dirs and pipeline_root is None:
        pipeline_root = os.path.join(cur, "src")
    if "train" in dirs and "test" in dirs and images_root is None:
        images_root = cur
        dirs[:] = [d for d in dirs if d not in ("train", "test")]

print()
print("Detected pipeline root (parent of src/):", pipeline_root)
print("Detected images root (has train/ and test/):", images_root)

assert pipeline_root is not None, (
    "Could not find a 'src' folder anywhere under /kaggle/input -- make sure "
    "the LATEST unifesp_pipeline.zip (with models_hybrid.py, hierarchy.py, "
    "losses_hierarchical.py, train_hybrid.py, evaluate_hybrid.py) was "
    "uploaded as a NEW VERSION of your pipeline dataset."
)
assert images_root is not None, (
    "Could not find a dataset with train/ and test/ folders -- attach the "
    "original felipekitamura/unifesp-xray-bodypart-classification dataset."
)
PIPELINE_DATASET = os.path.dirname(pipeline_root)
IMAGES_DATASET = images_root

Tree under /kaggle/input (dirs only, depth<=2, for debugging -- capped at 200 lines):
input/
  datasets/
    felipekitamura/
    noorabdullah02/

Detected pipeline root (parent of src/): /kaggle/input/datasets/noorabdullah02/felipekitamura/src
Detected images root (has train/ and test/): /kaggle/input/datasets/felipekitamura/unifesp-xray-bodypart-classification


In [ ]:
import shutil, sys

shutil.copytree(os.path.join(PIPELINE_DATASET, "src"), "/kaggle/working/src", dirs_exist_ok=True)
sys.path.insert(0, "/kaggle/working")

os.environ["UNIFESP_DATA_ROOT"] = IMAGES_DATASET
os.environ["UNIFESP_INDEX_ROOT"] = PIPELINE_DATASET
os.environ["UNIFESP_OUTPUT_DIR"] = "/kaggle/working/runs"
os.environ["UNIFESP_CACHE_DIR"] = "/kaggle/working/image_cache"

print("DATA_ROOT  ->", os.environ["UNIFESP_DATA_ROOT"])
print("INDEX_ROOT ->", os.environ["UNIFESP_INDEX_ROOT"])

# sanity: confirm the Step 3 files actually made it into this dataset version
for f in ["models_hybrid.py", "hierarchy.py", "losses_hierarchical.py", "train_hybrid.py", "evaluate_hybrid.py"]:
    print(f, "->", os.path.exists(os.path.join(PIPELINE_DATASET, "src", f)))

DATA_ROOT  -> /kaggle/input/datasets/felipekitamura/unifesp-xray-bodypart-classification
INDEX_ROOT -> /kaggle/input/datasets/noorabdullah02/felipekitamura
models_hybrid.py -> True
hierarchy.py -> True
losses_hierarchical.py -> True
train_hybrid.py -> True
evaluate_hybrid.py -> True


In [ ]:
!pip install -q timm albumentations iterative-stratification thop grad-cam

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())

CUDA available: True | GPU count: 2


2. Pre-cache DICOM -> PNG (skip if already cached from Step 2 in this session)

In [ ]:
!cd /kaggle/working && python -m src.cache_images

Indexing DICOM files under /kaggle/input/datasets/felipekitamura/unifesp-xray-bodypart-classification ...
Found 2481 files. Caching to /kaggle/working/image_cache ...
  250/2481 (23.6s elapsed)
  500/2481 (45.7s elapsed)
  750/2481 (67.8s elapsed)
  1000/2481 (90.0s elapsed)
  1250/2481 (112.2s elapsed)
  1500/2481 (134.1s elapsed)
  1750/2481 (156.3s elapsed)
  2000/2481 (178.5s elapsed)
  2250/2481 (200.6s elapsed)
Done. Cached 2481 images in 221.0s -> /kaggle/working/image_cache


3. Train both loss variants, one per GPU

plain = ordinary multilabel BCE (same as Step 2's loss). hierarchical = BCE + a group-level anatomical term (Head/Spine/Trunk/UpperLimb/LowerLimb/ Other). Running both side by side gives the Step 4 ablation for free.

--max_hours 10 leaves a ~2h buffer in the 12h session; the hybrid model is heavier than a single backbone so epochs take longer than Step 2's. If a session still cuts off, re-run this same cell in a new session -- it resumes from each variant's checkpoint_last.pt.

In [ ]:
import subprocess

def launch(loss_variant, gpu_id):
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    log = open(f"/kaggle/working/hybrid_{loss_variant}.log", "w")
    return subprocess.Popen(
        ["python", "-m", "src.train_hybrid", "--loss", loss_variant, "--fold", "0",
         "--epochs", "30", "--batch_size", "24", "--max_hours", "10"],
        cwd="/kaggle/working", env=env, stdout=log, stderr=subprocess.STDOUT,
    )

variants = ["plain", "hierarchical"]
procs = list(zip(variants, [launch(v, gpu) for gpu, v in enumerate(variants)]))

failed = []
for name, p in procs:
    rc = p.wait()
    if rc != 0:
        failed.append(name)
        print(f"[FAILED] hybrid_{name} exited with code {rc} -- check /kaggle/working/hybrid_{name}.log")
    else:
        print(f"[ok] hybrid_{name}")

if failed:
    print("\nFailed variants:", failed)
else:
    print("\nBoth hybrid variants finished successfully.")

[ok] hybrid_plain
[FAILED] hybrid_hierarchical exited with code 1 -- check /kaggle/working/hybrid_hierarchical.log

Failed variants: ['hierarchical']


Check progress any time (run in a new cell while training is in progress):

In [ ]:
!tail -n 30 /kaggle/working/hybrid_hierarchical.log

    main()
    ~~~~^^
  File "/kaggle/working/src/train_hybrid.py", line 78, in main
    history, best_metric = run_training(
                           ~~~~~~~~~~~~^
        model, train_loader, val_loader, optimizer, scheduler, loss_fn,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        device, run_dir, n_epochs=args.epochs, max_hours=args.max_hours,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/kaggle/working/src/engine.py", line 159, in run_training
    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device, scaler)
  File "/kaggle/working/src/engine.py", line 84, in train_one_epoch
    loss = loss_fn(logits, targets)
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1779, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.p

4. Evaluate: Step 3 hybrid vs Step 2 baselines, side by side

Uses the exact same sample_mean_f1 (Kaggle leaderboard metric) and macro_f1 as Step 2, so this is a direct, fair comparison -- not just "did the loss change", but "did we beat Swin-Tiny's 0.928 / DenseNet121's 0.875". Needs Step 2's leaderboard_comparable_fold0.csv to exist (from src.evaluate_checkpoints) for the side-by-side table; it still prints the hybrid-only numbers if that file isn't there.

In [ ]:
!cd /kaggle/working && python -m src.evaluate_hybrid --fold 0

indexing DICOM files under /kaggle/input/datasets/felipekitamura/unifesp-xray-bodypart-classification ...
[hybrid_plain] sample_mean_f1=0.9207  macro_f1=0.8677  (epoch 24)
[skip] hybrid_hierarchical: no checkpoint_best.pt in /kaggle/working/runs/hybrid_hierarchical_fold0

Saved: /kaggle/working/runs/hybrid_results_fold0.csv

(Run src.evaluate_checkpoints first to get /kaggle/working/runs/leaderboard_comparable_fold0.csv for a side-by-side comparison.)


In [ ]:
import pandas as pd
pd.read_csv("/kaggle/working/runs/step2_vs_step3_fold0.csv")

---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
/tmp/ipykernel_23/1655119165.py in <cell line: 0>()
      1 import pandas as pd
----> 2 pd.read_csv("/kaggle/working/runs/step2_vs_step3_fold0.csv")

/usr/local/lib/python3.13/dist-packages/pandas/io/parsers/readers.py in read_csv(filepath_or_buffer, sep, delimiter, header, names, index_col, usecols, dtype, engine, converters, true_values, false_values, skipinitialspace, skiprows, skipfooter, nrows, na_values, keep_default_na, na_filter, verbose, skip_blank_lines, parse_dates, infer_datetime_format, keep_date_col, date_parser, date_format, dayfirst, cache_dates, iterator, chunksize, compression, thousands, decimal, lineterminator, quotechar, quoting, doublequote, escapechar, comment, encoding, encoding_errors, dialect, on_bad_lines, delim_whitespace, low_memory, memory_map, float_precision, storage_options, dtype_backend)
   1024     kw

5. Before this session ends

/kaggle/working is wiped when the session closes unless you save it. Click "Save Version" now, or at minimum download runs/hybrid_plain_fold0/, runs/hybrid_hierarchical_fold0/, and the CSVs above.